# PLN — TF-IDF com Lematização e Stemming em Tweets #edtwt

Neste notebook, comparamos o impacto da **lematização** e **stemming** na representação TF-IDF.

Enquanto o TF-IDF vanilla (notebook `tdidf.ipynb`) aplica o vetorizador diretamente sobre o texto normalizado com regex, aqui investigamos o efeito de:
- **Lematização** (spaCy): reduz cada palavra à sua forma canônica (ex: "correndo" → "correr").
- **Stemming** (NLTK SnowballStemmer): remove afixos, gerando o radical da palavra (ex: "correndo" → "corr").

A combinação lematização + stemming produz tokens mais abstratos e potencialmente mais informativos para o TF-IDF.

O pipeline inclui:
1. **Pré-processamento**: lematização com spaCy + stemming com NLTK.
2. **Vetorização** com `TfidfVectorizer` sobre os tokens processados.
3. **Diagnóstico** da distribuição de similaridade.
4. **Heatmap**, clustering com KMeans, PCA e t-SNE.
5. **Comparação** direta com o TF-IDF vanilla (sem lematização/stemming adicional).

---

In [ ]:
%pip install scikit-learn plotly seaborn matplotlib pandas numpy nltk spacy --quiet

In [ ]:
import os
import json
import ast
import warnings
import numpy as np
import pandas as pd

import spacy
from nltk.stem import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

seed = 42

In [ ]:
# Download do modelo spaCy para portugues (pode levar alguns segundos)
!python -m spacy download pt_core_news_lg --quiet

## Carregamento dos Dados

Utilizamos o `entrega_2.csv`, que contém a coluna `normalizacao_re` com o texto já normalizado via regex.
Esse texto servirá como entrada para a pipeline de lematização + stemming.

In [ ]:
DATA_DIR = os.getcwd()
csv_path = os.path.join(DATA_DIR, '..', 'entregas', 'p2', 'entrega_2.csv')
json_path = os.path.join(DATA_DIR, '..', 'entregas', 'p2', 'entrega_2_tfidf_features.json')

if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Arquivo nao encontrado: {csv_path}')

df = pd.read_csv(csv_path)
print(f'Dataset: {df.shape[0]} tweets, {df.shape[1]} colunas')
print(f'Colunas relevantes: text, normalizacao_re, features_tfidf_sklearn')

## Pré-processamento: Lematização + Stemming

Aplicamos duas etapas consecutivas ao texto normalizado:

1. **Lematização (spaCy `pt_core_news_lg`)**: converte cada token para sua forma canônica (lemma).
   - Ex: "correndo" → "correr", "gatos" → "gato"
2. **Stemming (NLTK `SnowballStemmer('portuguese')`)**: reduz o lemma ao radical.
   - Ex: "correr" → "corr", "gato" → "gat"

O resultado é um texto com tokens normalizados morfologicamente, que serve de entrada para o `TfidfVectorizer`.

---

### Lematização vs. Stemming

- **Lematização**: usa conhecimento linguístico (dicionário, classe gramatical) para encontrar a forma base.
  - Mais precisa, mas mais lenta. Requer modelo de idioma.
- **Stemming**: aplica regras heurísticas de remoção de afixos.
  - Mais rápida, mas pode gerar radicais que não são palavras reais.

Ao combinar ambas, obtemos tokens compactos e semanticamente agrupados.

In [ ]:
nlp = spacy.load('pt_core_news_lg', disable=['ner', 'parser'])
stemmer = SnowballStemmer('portuguese')

def lematizar_e_stem(texto):
    """
    Aplica lematizacao com spaCy e stemming com NLTK.
    Retorna string com tokens processados, separados por espaco.
    """
    if not texto or not isinstance(texto, str) or not texto.strip():
        return ''
    doc = nlp(texto[:500])
    tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.strip()
        if lemma:
            stem = stemmer.stem(lemma.lower())
            if stem:
                tokens.append(stem)
    return ' '.join(tokens)

# Exemplo
exemplo = df['normalizacao_re'].iloc[0]
print(f'Original:  {exemplo[:120]}')
print(f'Processado: {lematizar_e_stem(exemplo)[:120]}')

In [ ]:
from tqdm.auto import tqdm

textos_normalizados = df['normalizacao_re'].fillna('').values

print('Aplicando lematizacao + stemming em todos os tweets...')
textos_processados = [lematizar_e_stem(t) for t in tqdm(textos_normalizados, desc='Processando')]

vazios = sum(1 for t in textos_processados if not t)
print(f'Tweets processados: {len(textos_processados)}')
print(f'Tweets sem tokens apos processamento: {vazios}')

## TF-IDF sobre Texto Lematizado + Stemmizado

Construímos o `TfidfVectorizer` com os mesmos hiperparâmetros do TF-IDF vanilla para garantir comparabilidade.

In [ ]:
vetorizador_lemmastem = TfidfVectorizer(
    lowercase=False,
    max_df=0.85,
    min_df=2,
    max_features=1000,
    token_pattern=r"(?u)\b\w\w+\b",
)

matriz_lemmastem = vetorizador_lemmastem.fit_transform(
    [t if t else ' ' for t in textos_processados]
)
vocabulario_lemmastem = vetorizador_lemmastem.get_feature_names_out()

print(f'Vocabulario: {len(vocabulario_lemmastem)} termos')
print(f'Matriz TF-IDF: {matriz_lemmastem.shape}')
print(f'Termos mais frequentes (TF-IDF): {vocabulario_lemmastem[np.argsort(matriz_lemmastem.sum(axis=0).A1)[-15:][::-1]]}')

In [ ]:
vetores_lemmastem = matriz_lemmastem.toarray().astype(np.float32)
documentos = df['text'].values

print(f'Vetores TF-IDF (lemmatizacao+stemming): {vetores_lemmastem.shape}')
print(f'Dimensao: {vetores_lemmastem.shape[1]}')
print(f'Densidade media: {(vetores_lemmastem > 0).mean():.2%}')

## Diagnóstico da Similaridade de Cosseno

In [ ]:
sim_raw = cosine_similarity(vetores_lemmastem)
upper_raw = sim_raw[np.triu_indices_from(sim_raw, k=1)]

print('=== Similaridade de Cosseno ORIGINAL (sem centralizacao) ===')
print(f'  Media: {upper_raw.mean():.6f}')
print(f'  Desvio padrao: {upper_raw.std():.6f}')
print(f'  Minimo: {upper_raw.min():.4f}  |  Maximo: {upper_raw.max():.4f}')
print(f'  Mediana: {np.median(upper_raw):.6f}')

In [ ]:
vetor_medio_global = vetores_lemmastem.mean(axis=0)
vetores_centralizados = vetores_lemmastem - vetor_medio_global

sim_cent = cosine_similarity(vetores_centralizados)
upper_cent = sim_cent[np.triu_indices_from(sim_cent, k=1)]

print('=== Similaridade de Cosseno CENTRALIZADA (mean-centering) ===')
print(f'  Media: {upper_cent.mean():.6f}')
print(f'  Desvio padrao: {upper_cent.std():.6f}')
print(f'  Minimo: {upper_cent.min():.4f}  |  Maximo: {upper_cent.max():.4f}')
print(f'  Mediana: {np.median(upper_cent):.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(upper_raw, bins=80, color='#4C72B0', edgecolor='white', alpha=0.9)
axes[0].axvline(upper_raw.mean(), color='red', linestyle='--', label=f'Media: {upper_raw.mean():.4f}')
axes[0].set_title('Original (sem centralizacao)', fontsize=12)
axes[0].set_xlabel('Similaridade de Cosseno')
axes[0].set_ylabel('Frequencia')
axes[0].legend()

axes[1].hist(upper_cent, bins=80, color='#DD8452', edgecolor='white', alpha=0.9)
axes[1].axvline(upper_cent.mean(), color='red', linestyle='--', label=f'Media: {upper_cent.mean():.4f}')
axes[1].set_title('Centralizada (mean-centering)', fontsize=12)
axes[1].set_xlabel('Similaridade de Cosseno')
axes[1].set_ylabel('Frequencia')
axes[1].legend()

plt.suptitle('Distribuicao da Similaridade entre Pares de Tweets — TF-IDF + Lematizacao + Stemming', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

## Heatmap de Similaridade Centralizada (Amostra)

In [ ]:
np.random.seed(seed)
n_amostra = min(20, len(documentos))
indices_amostra = np.sort(np.random.choice(len(documentos), size=n_amostra, replace=False))

matriz_amostra = sim_cent[np.ix_(indices_amostra, indices_amostra)]
labels = [str(i) for i in indices_amostra]

plt.figure(figsize=(12, 10))
sns.heatmap(
    matriz_amostra,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    linewidths=.5,
    xticklabels=labels,
    yticklabels=labels,
    vmin=-1, vmax=1
)
plt.title('Similaridade de Cosseno — TF-IDF + Lematizacao + Stemming', fontsize=14)
plt.xlabel('Indice do Tweet', fontsize=12)
plt.ylabel('Indice do Tweet', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print('Tweets da amostra:\n')
for i in indices_amostra:
    texto = documentos[i].replace('\n', ' ')[:90]
    print(f'  [{i}] {texto}{"..." if len(documentos[i]) > 90 else ""}')

## Agrupamento (Clustering) com KMeans

In [ ]:
num_clusters = 3

kmeans = KMeans(n_clusters=num_clusters, random_state=seed, n_init=10)
kmeans.fit(vetores_centralizados)

df_cluster = pd.DataFrame({'Tweet': documentos, 'Cluster': kmeans.labels_})

print(f'Agrupamento com {num_clusters} clusters:\n')
for c in range(num_clusters):
    cluster = df_cluster[df_cluster['Cluster'] == c]
    print(f'  Cluster {c} ({len(cluster)} tweets):')
    for s in cluster['Tweet'].head(3):
        print(f'    - {s.replace(chr(10), " ")[:100]}')
    print()

print('Top termos discriminativos por cluster:')
for c in range(num_clusters):
    mascara = kmeans.labels_ == c
    centroide = vetores_lemmastem[mascara].mean(axis=0)
    centroide_outros = vetores_lemmastem[~mascara].mean(axis=0)
    diff = centroide - centroide_outros
    top_indices = np.argsort(diff)[-10:][::-1]
    print(f'  Cluster {c}: {", ".join(vocabulario_lemmastem[i] for i in top_indices)}')
    print()

## Visualização 2D com PCA e t-SNE

In [ ]:
pca_2d = PCA(n_components=2, random_state=seed)
vetores_pca = pca_2d.fit_transform(vetores_centralizados)

var_exp = pca_2d.explained_variance_ratio_
print(f'Variancia explicada: PC1={var_exp[0]:.2%}, PC2={var_exp[1]:.2%}')
print(f'Variancia total explicada (2 PCs): {var_exp.sum():.2%}')

plot_df_pca = pd.DataFrame({
    'PCA 1': vetores_pca[:, 0],
    'PCA 2': vetores_pca[:, 1],
    'Cluster': df_cluster['Cluster'].astype(str),
    'Tweet': [t.replace('\n', ' ')[:60] + ('...' if len(t) > 60 else '') for t in documentos]
})

fig_pca = px.scatter(
    plot_df_pca, x='PCA 1', y='PCA 2', color='Cluster',
    title=f'PCA 2D: Clusters — TF-IDF + Lematizacao + Stemming ({num_clusters} clusters)',
    hover_data={'PCA 1': ':.2f', 'PCA 2': ':.2f', 'Cluster': True, 'Tweet': True},
    width=900, height=700
)
fig_pca.update_traces(marker=dict(size=4, opacity=0.7))
fig_pca.show()

In [ ]:
perplexity = min(30, len(documentos) // 5)
tsne = TSNE(n_components=2, random_state=seed, perplexity=perplexity, max_iter=1000)
vetores_tsne = tsne.fit_transform(vetores_centralizados)

plot_df_tsne = pd.DataFrame({
    't-SNE 1': vetores_tsne[:, 0],
    't-SNE 2': vetores_tsne[:, 1],
    'Cluster': df_cluster['Cluster'].astype(str),
    'Tweet': [t.replace('\n', ' ')[:60] + ('...' if len(t) > 60 else '') for t in documentos]
})

fig_tsne = px.scatter(
    plot_df_tsne, x='t-SNE 1', y='t-SNE 2', color='Cluster',
    title=f't-SNE: Clusters — TF-IDF + Lematizacao + Stemming ({num_clusters} clusters)',
    hover_data={'t-SNE 1': ':.2f', 't-SNE 2': ':.2f', 'Cluster': True, 'Tweet': True},
    width=900, height=700
)
fig_tsne.update_traces(marker=dict(size=4, opacity=0.7))
fig_tsne.show()

## Visualização 3D com PCA

In [ ]:
pca_3d = PCA(n_components=3, random_state=seed)
vetores_pca3 = pca_3d.fit_transform(vetores_centralizados)

var_exp3 = pca_3d.explained_variance_ratio_
print(f'Variancia explicada: PC1={var_exp3[0]:.2%}, PC2={var_exp3[1]:.2%}, PC3={var_exp3[2]:.2%}')
print(f'Variancia total explicada (3 PCs): {var_exp3.sum():.2%}')

pca3_df = pd.DataFrame({
    'PCA 1': vetores_pca3[:, 0],
    'PCA 2': vetores_pca3[:, 1],
    'PCA 3': vetores_pca3[:, 2],
    'Cluster': df_cluster['Cluster'].astype(str),
    'Tweet': [t.replace('\n', ' ')[:60] + ('...' if len(t) > 60 else '') for t in documentos]
})

fig_3d = px.scatter_3d(
    pca3_df,
    x='PCA 1', y='PCA 2', z='PCA 3',
    color='Cluster',
    title=f'PCA 3D: Clusters — TF-IDF + Lematizacao + Stemming ({num_clusters} clusters)',
    hover_data={'PCA 1': ':.2f', 'PCA 2': ':.2f', 'PCA 3': ':.2f', 'Cluster': True, 'Tweet': True},
    width=900, height=700
)
fig_3d.update_traces(marker=dict(size=3, opacity=0.7))
fig_3d.show()

## Comparação: TF-IDF Vanilla vs. TF-IDF + Lematização + Stemming

Comparamos as matrizes de similaridade do TF-IDF vanilla (pré-computado na Entrega 2) com o TF-IDF aplicado após lematização + stemming.

- **TF-IDF Vanilla**: texto normalizado com regex, sem processamento morfológico adicional.
- **TF-IDF + Lematização + Stemming**: texto lematizado (spaCy) e depois stemmizado (NLTK).

Ambos os conjuntos são centralizados antes da comparação.

In [ ]:
def parse_features_tfidf(feature_str):
    if pd.isna(feature_str):
        return {}
    try:
        return ast.literal_eval(feature_str)
    except (ValueError, SyntaxError):
        return {}

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        tfidf_config = json.load(f)
    nomes_features_vanilla = tfidf_config.get('feature_names', [])

    dicts_tfidf = df['features_tfidf_sklearn'].apply(parse_features_tfidf)
    vetores_tfidf_vanilla = np.zeros((len(df), len(nomes_features_vanilla)), dtype=np.float32)
    for i, d in enumerate(dicts_tfidf):
        for idx_f, feat_name in enumerate(nomes_features_vanilla):
            vetores_tfidf_vanilla[i, idx_f] = d.get(feat_name, 0.0)

    np.random.seed(seed)
    n_comp = min(200, len(vetores_lemmastem))
    idx_comp = np.sort(np.random.choice(len(vetores_lemmastem), size=n_comp, replace=False))

    vanilla_sample = vetores_tfidf_vanilla[idx_comp]
    lemmastem_sample = vetores_lemmastem[idx_comp]

    vanilla_sample_cent = vanilla_sample - vanilla_sample.mean(axis=0)
    lemmastem_sample_cent = lemmastem_sample - lemmastem_sample.mean(axis=0)

    sim_vanilla = cosine_similarity(vanilla_sample_cent)
    sim_lemmastem = cosine_similarity(lemmastem_sample_cent)

    vanilla_upper = sim_vanilla[np.triu_indices_from(sim_vanilla, k=1)]
    lemmastem_upper = sim_lemmastem[np.triu_indices_from(sim_lemmastem, k=1)]

    print(f'TF-IDF Vanilla (centralizado):              media={vanilla_upper.mean():.4f}, std={vanilla_upper.std():.4f}')
    print(f'TF-IDF + Lematizacao + Stemming (cent.):     media={lemmastem_upper.mean():.4f}, std={lemmastem_upper.std():.4f}')
    print(f'Correlacao entre matrizes: {np.corrcoef(sim_vanilla.flatten(), sim_lemmastem.flatten())[0,1]:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    sns.heatmap(sim_vanilla, annot=False, cmap='RdBu_r', center=0,
                xticklabels=False, yticklabels=False, ax=axes[0])
    axes[0].set_title('TF-IDF Vanilla — Similaridade Centralizada', fontsize=12)
    sns.heatmap(sim_lemmastem, annot=False, cmap='RdBu_r', center=0,
                xticklabels=False, yticklabels=False, ax=axes[1])
    axes[1].set_title('TF-IDF + Lemmatizacao + Stemming — Similaridade Centralizada', fontsize=12)
    plt.suptitle('Comparacao: TF-IDF Vanilla vs. TF-IDF + Lemmatizacao + Stemming (200 tweets)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f'Arquivo de metadados TF-IDF nao encontrado: {json_path}')
    print('Execute a entrega 2 antes para gerar os dados necessarios.')

## Exemplo: Efeito da Lematização + Stemming no Vocabulário

Mostramos como palavras diferentes são agrupadas no mesmo radical após o processamento.

In [ ]:
palavras_teste = ['correndo', 'correr', 'correu', 'corrida', 'corredor',
                  'comendo', 'comer', 'comeu', 'comida', 'comidao',
                  'magra', 'magro', 'magrinha', 'magrao', 'emagrecer']

print('Efeito da lematizacao + stemming:\n')
print(f'{"Original":<20} {"Lemma (spaCy)":<20} {"Stem (NLTK)":<20}')
print('-' * 60)
for palavra in palavras_teste:
    doc = nlp(palavra)
    lemma = doc[0].lemma_ if len(doc) > 0 else palavra
    stem = stemmer.stem(lemma.lower()) if lemma else palavra
    print(f'{palavra:<20} {lemma:<20} {stem:<20}')

## Observações

- A **lematização** reduz a dimensionalidade implícita ao agrupar formas flexionadas (singular/plural, conjugações verbais).
- O **stemming** vai além, agrupando palavras derivadas da mesma raiz (ex: "corrida" e "correr" → "corr").
- A combinação produz um vocabulário menor e mais denso, potencialmente melhorando a qualidade dos clusters.
- Trade-off: a agressividade do stemming pode unir palavras semanticamente distintas que compartilham o mesmo radical.
- Comparado ao TF-IDF vanilla, esta abordagem tende a gerar vetores com maior variância entre documentos.